# Tereguwami (ተርጓሚ) — End-to-End Multimodal ESL Translation Pipeline
### Continuous Sign Recognition, Translation, and Reverse-Channel 3D Avatar Production
**Author**: Segni Seyoum Negasa | EGATE Advanced AI/ML Track

In [ ]:
import sys
import os
import numpy as np

# Ensure repository root is on path
sys.path.append(os.path.abspath('../../'))

from data.geez_preprocessor import geez_preprocessor
from data.esl_dataset_loader import esl_dataset
from models.perception.keypoint_normalizer import keypoint_normalizer
from models.recognition.temporal_models import temporal_recognizer
from models.translation.gloss_free_transformer import continuous_translator
from models.translation.constrained_decoder import constrained_decoder
from models.production.progressive_transformer import avatar_production_engine

print("All Tereguwami core modules loaded successfully!")

## 1. Ge'ez Script Preprocessing and Orthographic Normalization (§4.4)
Eliminates archaic Ethiopic word separators and unifies homophone character variants.

In [ ]:
raw_geez_input = "ዶክተር፡ብርቱ፤የራስ-ምታት፡አለኝ።"
cleaned_geez = geez_preprocessor.clean(raw_geez_input)
print("Raw Input:    ", raw_geez_input)
print("Cleaned Text: ", cleaned_geez)

## 2. MediaPipe Holistic Keypoint Loading and Normalization (§8.1)
Loads continuous 543 3D landmark sequences and applies spatial shoulder centering.

In [ ]:
sample = esl_dataset[0]
raw_keypoints = sample["keypoints"]
print(f"Sample ID: {sample['sample_id']} | Domain: {sample['domain']}")
print(f"Raw Keypoints Shape: {raw_keypoints.shape} (Frames, Landmarks, 3D Coordinates)")

normalized_features = keypoint_normalizer.prepare_multimodal_tensor(raw_keypoints)
print(f"Normalized Feature Matrix Shape: {normalized_features.shape}")

## 3. Non-Manual Facial Grammar Extraction (§8.3)
Extracts eyebrow elevation, eye aperture, mouth gestures, and head tilt/shake.

In [ ]:
facial_markers = keypoint_normalizer.extract_non_manual_features(raw_keypoints)
print("Eyebrow Elevation (AU1/2):", np.round(facial_markers["eyebrow_elevation"][:5], 3))
print("Mouth Aperture:           ", np.round(facial_markers["mouth_aperture"][:5], 3))
print("Head Lateral Tilt (deg):  ", np.round(np.degrees(facial_markers["head_tilt"][:5]), 2))

## 4. Continuous Gloss-Free Translation & Constrained Decoding (§8.4)
Translates temporal sign sequence into target text with safety guardrails.

In [ ]:
translation = continuous_translator.translate(
    keypoint_features=normalized_features,
    target_lang="am",
    domain_hint="healthcare"
)

guarded = constrained_decoder.decode_with_constraints(
    candidate_text=translation["translated_text"],
    confidence_score=translation["confidence_score"],
    recognized_glosses=[sample["gloss"]],
    domain="healthcare"
)

print("Translated Text:  ", guarded["final_text"])
print("Confidence Score: ", guarded["confidence_score"])
print("Is Faithful:      ", guarded["is_faithful"])
print("Requires Review:  ", guarded["requires_human_verification"])

## 5. Reverse-Channel 3D Avatar Generation (§8.5)
Synthesizes continuous avatar pose sequences from text.

In [ ]:
reply_prompt = "መድኃኒቱን ከምግብ በኋላ ይውሰዱ።"
avatar_stream = avatar_production_engine.generate_avatar_stream(
    text_input=reply_prompt,
    source_lang="am",
    signing_speed=1.0
)

print(f"Generated Avatar Performance for: '{reply_prompt}'")
print(f"Total Frames: {avatar_stream['total_frames']} | Duration: {avatar_stream['duration_seconds']}s")
print("Sample Frame 0 Blendshapes:", avatar_stream["frames"][0]["blendshapes"])